In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce").fillna(0.0)
df["y"] = (df["Churn"] == "Yes").astype(int)

num_features = ["tenure", "MonthlyCharges"]
cat_features = [
    "gender", "Contract", "PaymentMethod", "PaperlessBilling",
    "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
]
X_B = df[num_features + cat_features]
y = df["y"]

In [2]:
pipeline_B = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_features),
    ])),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
dev = df["tenure"] > 12
X_dev, y_dev = X_B[dev], y[dev]
X_new, y_new = X_B[~dev], y[~dev]
print(f"Development: {len(y_dev)} | Yeni: {len(y_new)}")
print(f"Churn faizi — Development: {y_dev.mean():.3f} | Yeni: {y_new.mean():.3f}")

Development: 4857 | Yeni: 2186
Churn faizi — Development: 0.171 | Yeni: 0.474


In [8]:
p_dev = cross_val_predict(pipeline_B, X_dev, y_dev, cv=cv, method="predict_proba")[:, 1]

pipeline_B.fit(X_dev, y_dev)
p_new = pipeline_B.predict_proba(X_new)[:, 1]

In [9]:
thr = np.quantile(p_dev, 0.90)

def report(y_true, p, thr):
    auc = roc_auc_score(y_true, p)
    pred = (p >= thr).astype(int)
    return {
        "AUC": round(auc, 4),
        "Gini": round(2 * auc - 1, 4),
        "Precision": round(precision_score(y_true, pred), 4),
        "Recall": round(recall_score(y_true, pred), 4),
    }

print("Hədd:", round(thr, 4))
print("Development:", report(y_dev, p_dev, thr))
print("Yeni       :", report(y_new, p_new, thr))

Hədd: 0.4704
Development: {'AUC': 0.8319, 'Gini': 0.6639, 'Precision': 0.5679, 'Recall': 0.3317}
Yeni       : {'AUC': 0.7684, 'Gini': 0.5368, 'Precision': 0.733, 'Recall': 0.54}


In [10]:
def psi(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    e = np.clip(e, 1e-6, None)
    a = np.clip(a, 1e-6, None)
    return np.sum((a - e) * np.log(a / e))

psi_mc = psi(df.loc[dev, "MonthlyCharges"], df.loc[~dev, "MonthlyCharges"])
psi_score = psi(p_dev, p_new)
print(f"PSI MonthlyCharges: {psi_mc:.4f}")
print(f"PSI skor          : {psi_score:.4f}")

PSI MonthlyCharges: 0.5201
PSI skor          : 1.8556


In [11]:
summary = pd.DataFrame({
    "Development": report(y_dev, p_dev, thr),
    "Yeni": report(y_new, p_new, thr),
}).T
print(summary)

                AUC    Gini  Precision  Recall
Development  0.8319  0.6639     0.5679  0.3317
Yeni         0.7684  0.5368     0.7330  0.5400


In [12]:
# 1) Kontrol PSI: eyni development, fərqli fold bölgüsü. Nəticə ~0 olmalıdır.
cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
p_dev_2 = cross_val_predict(pipeline_B, X_dev, y_dev, cv=cv2, method="predict_proba")[:, 1]
print("Kontrol PSI (eyni populyasiya):", round(psi(p_dev, p_dev_2), 4))

# 2) Hədd üzrə flag payı və baza churn
print("Flag faizi — dev:", round((p_dev >= thr).mean(), 4), "| yeni:", round((p_new >= thr).mean(), 4))
print("Churn faizi — dev:", round(y_dev.mean(), 3), "| yeni:", round(y_new.mean(), 3))

# 3) MonthlyCharges ortalamaları (PSI-nin nə üçün böyük olduğunu göstərir)
print("MonthlyCharges ort — dev:", round(df.loc[dev, "MonthlyCharges"].mean(), 2),
      "| yeni:", round(df.loc[~dev, "MonthlyCharges"].mean(), 2))

Kontrol PSI (eyni populyasiya): 0.001
Flag faizi — dev: 0.1001 | yeni: 0.3495
Churn faizi — dev: 0.171 | yeni: 0.474
MonthlyCharges ort — dev: 68.66 | yeni: 56.1


# ## Nəticələr (Faza 8)

**Bölmə.** Telco-da tarix yoxdur. Development: tenure > 12, yeni: tenure ≤ 12. Bu real out-of-time deyil, populyasiya dəyişməsinin proxy-sidir.

**Metrikalar.** Hədd development-dən top 10% kimi seçildi (0.4704). Development: AUC 0.8319, Gini 0.6639, Precision 0.5679, Recall 0.3317. Yeni: AUC 0.7684, Gini 0.5368, Precision 0.7330, Recall 0.5400.

**Şərh.** AUC və Gini yeni nümunədə zəifləyir, bu, modelin tenure ≤ 12 müştərilərdə sıralamada daha az güclü olduğunu göstərir. Precision və recall-un artması baza churn-in yüksək olmasından gəlir, modelin yaxşılaşmasından yox. Buna görə əsas müqayisə AUC/Gini-dir.

**PSI.** MonthlyCharges üzrə PSI = 0.5201 (ciddi dəyişiklik, sənaye eşiqləri ilə 0.25-dən yuxarı). Yeni müştərilərin ödəniş profili development-dən fərqlidir. Skor üzrə PSI = [kontrol nəticəsindən sonra yaz].

**Məhdudiyyət.** Proxy bölmə tenure ilə birbaşa bağlıdır, buna görə fərq modeldən çox populyasiya dəyişməsini əks etdirir. PSI eşikləri sənaye təcrübəsidir, tənzimləyici tələb deyil. Real monitorinq üçün zamana görə data lazımdır.